# 🌐 Atmosphäre-Analyse
## Layer 7 – Earth Field State Engine

| Layer | Name | Status |
|-------|------|--------|
| 0–6 | Datenlayer | ✅ vollständig |
| **7** | **Earth Field State Engine** | **← dieser Layer** |
| 8 | Research / Hypothesen | ⬜ |

> **Layer 7 ist KEIN Analyse-Layer.** Layer 7 ist eine **Engine**:
> Sie nimmt alle Outputs aus Layer 0–6 und erzeugt einen **vergleichbaren, zeitbasierten, maschinenlesbaren Earth-Field-Systemzustand** für Layer 8.

**Funktionen der Engine:**

0. Führt Layer 0–6 automatisch aus
1. Sammelt alle layer{0..6}_state.json
2. Normalisiert Scores, Levels, Confidence
3. Berechnet Trends (Δ1h, Δ6h, Δ24h, Volatilität) aus Historie
4. Berechnet Meta-Scores (downstream, preparation, activation, resonance)
5. Berechnet Cavity Gate (Typ + Score aus L4/L6)
6. Berechnet Layer-Kopplungen (Stärke, Lag, Confidence) – 7 Pfade
7. Berechnet Field Operators + Operator-Vektor
8. Bestimmt dominante / sekundäre / ruhige Layer
9. Klassifiziert Systemzustand (8 Klassen)
10. Erzeugt Tags (baseline_tags, signal_tags, event_tags)
11. Bereitet Layer 8 Handoff vor (Summary + Forschungsfragen)
12. Speichert aktuellen Zustand als layer7_state.json
13. Hängt Snapshot an layer7_history.jsonl an

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math, os
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# --- Projektpfade (CWD-unabhaengig, ohne pip install) ---
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / '.project-root').exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / 'src'))
from atmosphere.paths import layer_state, HISTORY

RUN_TIMESTAMP = datetime.datetime.utcnow().isoformat() + 'Z'
print(f'Engine-Lauf: {RUN_TIMESTAMP}')

HISTORY_FILE = HISTORY
STATE_FILE   = layer_state(7)

# Schwellenwerte
LEVEL_THRESHOLDS = {'ruhig': 0.30, 'moderat': 0.60, 'aktiv': 0.80}

def classify_level(score):
    if score is None: return 'unbekannt'
    if score < LEVEL_THRESHOLDS['ruhig']:   return 'ruhig'
    if score < LEVEL_THRESHOLDS['moderat']: return 'moderat'
    if score < LEVEL_THRESHOLDS['aktiv']:   return 'aktiv'
    return 'stark'

---
## 1. Sammeln aller Layer-Zustände

In [ ]:
# ============================================================
# Vorgaenger-States kommen vom Pipeline-Runner (run_pipeline.py),
# der L0..L6 GENAU EINMAL der Reihe nach ausfuehrt und ihre States
# schreibt. L7 fuehrt die Vorgaenger NICHT selbst aus — es liest nur
# deren layer_state(n) (siehe naechste Zelle). Das vermeidet einen
# zweiten, abweichenden Datenabruf und haelt einen einzigen Datenstand
# pro Lauf (Provenance-Disziplin).
#
# L7 standalone aktualisieren?  ->  python pipeline/run_pipeline.py --from 0
# ============================================================


In [ ]:
# ============================================================
# ALLE LAYER-ZUSTÄNDE EINLESEN
# ============================================================

layers = {}
for n in range(7):
    fp = layer_state(n)
    if os.path.exists(fp):
        try:
            with open(fp, encoding='utf-8') as f:
                layers[n] = json.load(f)
        except Exception as e:
            print(f'  L{n}: Fehler {e}')
            layers[n] = None
    else:
        print(f'  L{n}: Datei fehlt')
        layers[n] = None

print('LAYER-ZUSTANDS-ÜBERSICHT')
print('=' * 78)
print(f'  {"Layer":<6} {"Score":>7} {"Level":>10} {"Conf":>6} {"Dominant":<32}')
print('-' * 78)
for n, L in layers.items():
    if L:
        score = L.get('score', 0) or 0
        level = L.get('level', 'unbekannt')
        conf  = L.get('confidence', 0) or 0
        dom   = L.get('dominant_component') or L.get('dominant_driver') or '–'
        print(f'  L{n:<5} {score:>7.3f} {level:>10} {conf:>6.0%} {dom[:30]:<32}')
    else:
        print(f'  L{n:<5} {"–":>7} {"missing":>10} {"–":>6} {"–":<32}')
print('=' * 78)

---
## 2. Normalisierung: einheitlicher Layer-State pro Layer

In [ ]:
# ============================================================
# JEDER LAYER WIRD AUF GLEICHE STRUKTUR NORMALISIERT
# {score, level, confidence, dominant_component, flags, key_metrics}
# ============================================================

LAYER_NAMES = {
    0: 'L0_external_drivers',
    1: 'L1_planetary_body',
    2: 'L2_surface_zone',
    3: 'L3_atmosphere',
    4: 'L4_ionosphere',
    5: 'L5_global_electric_circuit',
    6: 'L6_resonance_field',
}

def extract_key_metrics(n, L):
    """Extrahiert die wichtigsten Kennzahlen pro Layer"""
    if not L: return {}
    rv = L.get('raw_values', {})
    if n == 0:
        return {'F10.7': rv.get('F10.7_sfu', {}).get('value') if isinstance(rv.get('F10.7_sfu'), dict) else rv.get('F10.7_sfu'),
                'Kp':    rv.get('Kp_index', {}).get('value') if isinstance(rv.get('Kp_index'), dict) else rv.get('Kp_index'),
                'IMF_Bz': rv.get('IMF_Bz_nT', {}).get('value') if isinstance(rv.get('IMF_Bz_nT'), dict) else rv.get('IMF_Bz_nT')}
    if n == 1:
        return {'seismic_events': rv.get('seismic_events_7d'),
                'max_mag':        rv.get('seismic_max_mag'),
                'LOD_anomaly':    rv.get('LOD_anomaly_ms')}
    if n == 2:
        enso = rv.get('ENSO', {})
        return {'SST_anomaly':  rv.get('SST_anomaly_degC'),
                'ENSO_phase':   enso.get('phase_observed'),
                'conv_pot':     rv.get('conv_potential_mean')}
    if n == 3:
        return {'CAPE':         rv.get('CAPE_mean_Jkg'),
                'thunder_pts':  rv.get('thunder_points_WMO'),
                'storms_EONET': rv.get('storm_events_EONET')}
    if n == 4:
        rs = L.get('resonance_system', {})
        return {'Kp':         rv.get('Kp_current'),
                'cavity_h':   rs.get('cavity_height_km'),
                'xray_class': rv.get('xray_class')}
    if n == 5:
        gec = L.get('gec_state', {})
        return {'V_iono':         gec.get('V_ionosphere_kV'),
                'delta_V_pct':    gec.get('delta_V_pct'),
                'generator':      gec.get('generator_strength')}
    if n == 6:
        em = L.get('expected_modulated', {}).get('SR_1', {})
        da = L.get('delta_analysis', {})
        return {'SR1_Hz':           em.get('freq_Hz'),
                'SR1_amp_pT':       em.get('amplitude_pT'),
                'non_geom_ratio':   da.get('ratio_non_geom_to_geom')}
    return {}

normalized = {}
for n in range(7):
    L = layers.get(n)
    if not L:
        normalized[LAYER_NAMES[n]] = {
            'score': None, 'level': 'unbekannt', 'confidence': 0.0,
            'dominant_component': None, 'flags': {}, 'key_metrics': {},
            'available': False,
        }
        continue
    score = L.get('score')
    normalized[LAYER_NAMES[n]] = {
        'score':              score,
        'level':              classify_level(score) if score is not None else L.get('level', 'unbekannt'),
        'confidence':         L.get('confidence', 0.0) or 0.0,
        'dominant_component': L.get('dominant_component') or L.get('dominant_driver'),
        'flags':              L.get('flags', {}),
        'key_metrics':        extract_key_metrics(n, L),
        'available':          True,
    }

available_layers = [k for k, v in normalized.items() if v['available']]
print(f'Verfügbare Layer: {len(available_layers)}/7')
for name, st in normalized.items():
    if st['available']:
        print(f'  {name:<32} score={st["score"]:.3f}  conf={st["confidence"]:.0%}  level={st["level"]}')

# ============================================================
# ENSO KONTEXT — liest direkt aus Layer 2
# ============================================================

def classify_enso(L2):
    if not L2:
        return {'phase_class': 'unknown', 'macro_phase': 'unknown',
                'direction': 'unknown', 'event_risk': 'unknown',
                'nino34_anomaly_degC': None, 'oni_3month_degC': None,
                'used_in_score': False, 'source': 'no_data'}

    enso = L2.get('raw_values', {}).get('ENSO', {})
    return {
        'phase_class':         enso.get('phase_class', 'unknown'),
        'macro_phase':         enso.get('macro_phase', 'unknown'),
        'direction':           enso.get('direction', 'unknown'),
        'event_risk':          enso.get('event_risk', 'unknown'),
        'nino34_anomaly_degC': enso.get('weekly_nino34_anomaly_degC'),
        'oni_3month_degC':     enso.get('oni_3month_degC'),
        'used_in_score':       enso.get('phase_class', 'unknown') != 'unknown',
        'source':              enso.get('source', 'unknown'),
    }

enso_context = classify_enso(layers.get(2))
print(f'ENSO: {enso_context["phase_class"]}  ({enso_context["event_risk"]})  '
      f'Niño3.4={enso_context["nino34_anomaly_degC"]}°C  '
      f'ONI={enso_context["oni_3month_degC"]}°C  '
      f'Richtung={enso_context["direction"]}')

In [ ]:
# ============================================================
# LAYER 2.5 (MESO) — Organisation × CIN-Gate
# Liest CIN/Cloud aus L3 (bereits vorhandene Werte, kein neuer Feed),
# versucht echtes OLR (PSL); faellt bei Veralterung auf den Wolken-
# Uebergangsproxy zurueck (s. meso_ingest.py). Persistiert eigenstaendig
# UND haengt sich in die normale Layer-Normalisierung ein, damit Trends,
# History und der Layer-8-Handoff das Meso-Feld automatisch mitfuehren.
# ============================================================

from atmosphere.ingest.meso_ingest import (
    ingest_meso, load_cin_from_l3_state, load_cloud_from_l3_state, load_olr_anomaly,
)

l3_raw = layers.get(3)
if l3_raw:
    _meso_cin   = load_cin_from_l3_state(l3_raw)
    _meso_cloud = load_cloud_from_l3_state(l3_raw)
    meso_result = ingest_meso(olr_loader=load_olr_anomaly, cin_from_l3=_meso_cin, cloud_cover=_meso_cloud)
else:
    meso_result = None
    print('  L2.5 (Meso): L3-State fehlt \u2014 Meso wird uebersprungen')

if meso_result:
    meso_result['timestamp']      = RUN_TIMESTAMP
    meso_result['engine_version'] = '1.0'
    meso_result['layer']          = '2.5'
    meso_result['name']           = 'Meso-Scale Convective Organisation'

    # Ehrlichkeits-Trennung (s. Evidence-Consistency-Reparaturliste P1.5):
    # source_status = kam ueberhaupt ein realer Dateneingang an (measured/inferred).
    # construct_status = ist das theoretische Konstrukt "organisierte Konvektion"
    # DIREKT gemessen (nur OLR) oder nur ueber einen schwaecheren, aber realen
    # Ersatzindikator angenaehert (cloud_proxy)? Gesamtbewoelkung ist ein Proxy
    # fuer Organisation, keine direkte Messung davon.
    _org_src = meso_result['ingest'].get('organisation_source')
    meso_result['source_status']      = meso_result['holon_status']       # 'measured' | 'inferred'
    meso_result['construct_status']   = (
        'measured'      if _org_src == 'olr'
        else 'derived_proxy' if _org_src == 'cloud_proxy'
        else 'inferred'
    )
    meso_result['evidence_status']    = meso_result['construct_status']  # Alias fuer Doku/Report-Konsumenten
    meso_result['shared_source_with'] = ['L3_atmosphere']   # CIN + Cloud-Proxy kommen aus L3 (Confound-Hinweis)

    with open(layer_state('2p5_meso'), 'w', encoding='utf-8') as f:
        json.dump(meso_result, f, indent=2, ensure_ascii=False)
    print(f"\u2705 {layer_state('2p5_meso')} gespeichert")
    print(f"   source_status={meso_result['source_status']}  construct_status={meso_result['construct_status']}  "
          f"organisation_source={_org_src}  score={meso_result['score']}")

    normalized['L2p5_meso'] = {
        'score':              meso_result['score'],
        'level':              meso_result['level'],
        'confidence':         meso_result['confidence'],
        'dominant_component': meso_result['dominant_component'],
        'flags':              meso_result['flags'],
        'key_metrics': {
            'organisation':        meso_result['organisation'],
            'cin_gate':            meso_result['cin_gate'],
            'organisation_source': _org_src,
            'construct_status':    meso_result['construct_status'],
            'CIN':                 meso_result['key_metrics'].get('CIN'),
            'cloud_cover':         meso_result['key_metrics'].get('cloud_cover'),
        },
        'available':          bool(meso_result.get('available')),
    }
else:
    normalized['L2p5_meso'] = {
        'score': None, 'level': 'unbekannt', 'confidence': 0.0,
        'dominant_component': None, 'flags': {}, 'key_metrics': {},
        'available': False,
    }

print(f"  L2p5_meso                       score={normalized['L2p5_meso']['score']}  "
      f"level={normalized['L2p5_meso']['level']}  "
      f"construct={normalized['L2p5_meso']['key_metrics'].get('construct_status')}")


---
## 3. Trend-Analyse aus Historie

In [ ]:
# ============================================================
# HISTORIE EINLESEN UND DELTAS BERECHNEN
# ============================================================

history = []
if os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    history.append(json.loads(line))
                except: continue

print(f'Historie: {len(history)} Snapshots geladen')

def find_snapshot_at_age(history, hours_ago, tolerance_hours=2):
    """Findet Snapshot der ~hours_ago alt ist"""
    target = datetime.datetime.utcnow() - datetime.timedelta(hours=hours_ago)
    best = None; best_diff = float('inf')
    for snap in history:
        try:
            t = datetime.datetime.fromisoformat(snap['timestamp'].replace('Z', ''))
            diff = abs((t - target).total_seconds() / 3600)
            if diff < best_diff and diff < tolerance_hours:
                best = snap; best_diff = diff
        except: continue
    return best

snap_1h  = find_snapshot_at_age(history, 1,  tolerance_hours=1.5)
snap_6h  = find_snapshot_at_age(history, 6,  tolerance_hours=3)
snap_24h = find_snapshot_at_age(history, 24, tolerance_hours=6)

trends = {}
for name, st in normalized.items():
    if not st['available'] or st['score'] is None:
        trends[name] = {'delta_1h': None, 'delta_6h': None, 'delta_24h': None,
                        'trend': 'unknown', 'volatility': None}
        continue
    cur = st['score']
    d_1h  = round(cur - snap_1h['layers'][name]['score'], 4)  if snap_1h  and snap_1h.get('layers',{}).get(name,{}).get('score') is not None else None
    d_6h  = round(cur - snap_6h['layers'][name]['score'], 4)  if snap_6h  and snap_6h.get('layers',{}).get(name,{}).get('score') is not None else None
    d_24h = round(cur - snap_24h['layers'][name]['score'], 4) if snap_24h and snap_24h.get('layers',{}).get(name,{}).get('score') is not None else None

    # Trend-Kategorie aus 6h-Delta (oder 24h falls 6h fehlt)
    delta_for_trend = d_6h if d_6h is not None else d_24h
    if delta_for_trend is None:
        trend = 'unknown'
    elif delta_for_trend > 0.05: trend = 'rising'
    elif delta_for_trend < -0.05: trend = 'falling'
    else: trend = 'stable'

    # Volatilität: Standardabweichung der letzten 10 Snapshots
    recent_scores = [s['layers'].get(name,{}).get('score') for s in history[-10:]
                     if s.get('layers',{}).get(name,{}).get('score') is not None]
    volatility = round(float(np.std(recent_scores)), 4) if len(recent_scores) >= 3 else None

    trends[name] = {
        'delta_1h':   d_1h,
        'delta_6h':   d_6h,
        'delta_24h':  d_24h,
        'trend':      trend,
        'volatility': volatility,
    }

print('\nTrend-Übersicht:')
for name, t in trends.items():
    if normalized[name]['available']:
        d1  = f'{t["delta_1h"]:+.3f}'  if t['delta_1h']  is not None else '   –   '
        d6  = f'{t["delta_6h"]:+.3f}'  if t['delta_6h']  is not None else '   –   '
        d24 = f'{t["delta_24h"]:+.3f}' if t['delta_24h'] is not None else '   –   '
        v   = f'{t["volatility"]:.3f}' if t['volatility'] is not None else '   –   '
        print(f'  {name:<32}  Δ1h={d1}  Δ6h={d6}  Δ24h={d24}  vol={v}  [{t["trend"]}]')

---
## 4. Kopplungs-Berechnung zwischen Layern

In [ ]:
# ============================================================
# KOPPLUNGS-MATRIX
# Bekannte physikalische Kopplungen mit Stärke aus den Daten
# ============================================================

def safe(v, default=0.0):
    return float(v) if v is not None else default

# Hilfsabfragen
L0 = layers.get(0); L1 = layers.get(1); L2 = layers.get(2)
L3 = layers.get(3); L4 = layers.get(4); L5 = layers.get(5); L6 = layers.get(6)

# Definierte Kopplungen mit Berechnungslogik
couplings = []

# L0 → L4 (Space Weather → Ionosphäre)
if L0 and L4:
    s0 = safe(L0.get('score'))
    s4 = safe(L4.get('score'))
    # Stärke: korrelierter Anteil
    strength = round(min(s0, s4) * 0.9 + abs(s0 - s4) * 0.1, 3)
    couplings.append({
        'from': 'L0_external_drivers', 'to': 'L4_ionosphere',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L0.get('confidence',0), L4.get('confidence',0)), 2),
        'mechanism': 'Solarwind, F10.7, X-Ray ionisieren Ionosphäre'
    })

# L3 → L5 (Gewitter → GEC Generator)
if L3 and L5:
    thunder_score = safe(L3.get('components',{}).get('Gewitteraktivität',{}).get('score'))
    generator     = safe(L5.get('gec_state',{}).get('generator_strength', 1.0)) - 1.0
    strength = round(min(1.0, abs(generator) * 2 + thunder_score * 0.5), 3)
    couplings.append({
        'from': 'L3_atmosphere', 'to': 'L5_global_electric_circuit',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L3.get('confidence',0), L5.get('confidence',0)), 2),
        'mechanism': 'Gewitter laden Ionosphäre (CAPE → V_iono)'
    })

# L3 → L6 (Blitzaktivität → Schumann-Amplitude)
if L3 and L6:
    thunder_score = safe(L3.get('components',{}).get('Gewitteraktivität',{}).get('score'))
    amp_factor    = safe(L6.get('modulators',{}).get('amplitude_factor',{}).get('thunder_l3', 1.0)) - 1.0
    strength = round(min(1.0, thunder_score * 0.7 + abs(amp_factor) * 3), 3)
    couplings.append({
        'from': 'L3_atmosphere', 'to': 'L6_resonance_field',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L3.get('confidence',0), L6.get('confidence',0)), 2),
        'mechanism': 'Blitze regen Schumann-Resonanz an'
    })

# L4 → L6 (Cavity → Resonanzbedingungen)
if L4 and L6:
    cav_dev = abs(safe(L4.get('resonance_system',{}).get('cavity_delta_km'))) / 20
    ioniz   = safe(L4.get('components',{}).get('Ionisierungsgrad (F10.7)',{}).get('score'))
    strength = round(min(1.0, cav_dev * 0.5 + ioniz * 0.5), 3)
    couplings.append({
        'from': 'L4_ionosphere', 'to': 'L6_resonance_field',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L4.get('confidence',0), L6.get('confidence',0)), 2),
        'mechanism': 'Cavity-Höhe + Leitfähigkeit modulieren Frequenz und Q'
    })

# L5 → L6 (GEC-Potential → Resonanz-Energie)
if L5 and L6:
    delta_v_pct = abs(safe(L5.get('gec_state',{}).get('delta_V_pct'))) / 30
    s5 = safe(L5.get('score'))
    strength = round(min(1.0, delta_v_pct * 0.6 + s5 * 0.4), 3)
    couplings.append({
        'from': 'L5_global_electric_circuit', 'to': 'L6_resonance_field',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L5.get('confidence',0), L6.get('confidence',0)), 2),
        'mechanism': 'GEC ist die elektrische Architektur der Resonanz'
    })

# L0 → L5 (Geomagnetische Modulation des GEC)
if L0 and L5:
    kp_score = safe(L0.get('components',{}).get('Geomagn. Aktivität (Kp)'))
    s5 = safe(L5.get('score'))
    strength = round(min(1.0, kp_score * 0.6 + s5 * 0.2), 3)
    couplings.append({
        'from': 'L0_external_drivers', 'to': 'L5_global_electric_circuit',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L0.get('confidence',0), L5.get('confidence',0)), 2),
        'mechanism': 'Kp moduliert ionosphärische Leitfähigkeit (GEC-Widerstand)'
    })

# L2 → L3 (Konvektion → Gewitter)
if L2 and L3:
    conv = safe(L2.get('components',{}).get('Konvektions-Potential',{}).get('score'))
    s3 = safe(L3.get('score'))
    strength = round(min(1.0, conv * 0.7 + s3 * 0.3), 3)
    couplings.append({
        'from': 'L2_surface_zone', 'to': 'L3_atmosphere',
        'strength': strength, 'lag_hours': 1,
        'confidence': round(min(L2.get('confidence',0), L3.get('confidence',0)), 2),
        'mechanism': 'Oberflächenkonvektion → atmosphärische Gewitterbildung'
    })

print('KOPPLUNGS-MATRIX')
print('=' * 78)
for c in couplings:
    arrow = ('▰' * int(c['strength']*10) + '▱' * (10 - int(c['strength']*10)))
    print(f'  {c["from"]:<28} → {c["to"]:<26}  {arrow}  {c["strength"]:.3f}')
print('=' * 78)

---
## 5. Dominante Layer & System-State-Klassifikation

In [ ]:
# ============================================================
# DOMINANTE LAYER ERKENNEN
# ============================================================

# Layer, die NICHT in schichtuebergreifende Aggregate eingehen.
#
# L2p5_meso ist ein MEDIATOR-Proxy (organisation x cin_gate), kein physischer
# Messlayer der Kette L0..L6. Zwei Gruende fuer den Ausschluss:
#
#  1) Zeitreihen-Bruch: avg_score ist eine HISTORISCHE Trendgroesse, die L8
#     auswertet. Waechst die Layer-Menge mitten in der Reihe von 7 auf 8, springt
#     der Mittelwert am Integrationsdatum -- eine reine Schema-Aenderung, die wie
#     ein physikalisches Signal aussieht (bei meso=0.0 rund -12%, bei 0.7 +15%).
#  2) Klassifikator-Kontamination: aus `scored` entstehen dominant_layer und
#     active_layers, und classify_system_state entscheidet mit `len(active) >= 3`.
#     Ein achter Kandidat verschiebt diese Schwelle still -- der Meso-Score kann
#     seit der konvektiven Aggregation durchaus >= 0.5 erreichen.
#
# Meso geht dadurch NICHT verloren: es steht vollstaendig unter
# layers.L2p5_meso im State und in der History. Es wird nur nicht in Mittelwerte
# ueber inkommensurable Groessen hineingerechnet.
AGGREGATE_EXCLUDE = {'L2p5_meso'}

scored = [(name, st['score']) for name, st in normalized.items()
          if st['available'] and st['score'] is not None
          and name not in AGGREGATE_EXCLUDE]
scored_sorted = sorted(scored, key=lambda x: -x[1])

dominant_layer  = scored_sorted[0][0]   if len(scored_sorted) >= 1 else None
secondary_layer = scored_sorted[1][0]   if len(scored_sorted) >= 2 else None
weak_layers     = [n for n, s in scored_sorted if s < 0.3]
active_layers   = [n for n, s in scored_sorted if s >= 0.5]

print(f'Dominanter Layer:  {dominant_layer}  ({normalized[dominant_layer]["score"]:.3f})')
print(f'Sekundärer Layer:  {secondary_layer} ({normalized[secondary_layer]["score"]:.3f})' if secondary_layer else '')
print(f'Aktive Layer:      {active_layers}')
print(f'Ruhige Layer:      {weak_layers}')

# ============================================================
# META-SCORES — Aggregierte Layer-Indikatoren
# Macht State-Klassifikation unabhängiger von einzelnen Layern
# ============================================================

def _s(name):
    """Safe-Score-Lookup aus normalized"""
    v = normalized.get(name, {}).get('score')
    return float(v) if v is not None else 0.0

meta_scores = {
    'preparation_score': _s('L2_surface_zone'),
    'activation_score':  _s('L3_atmosphere'),
    'electric_score':    _s('L5_global_electric_circuit'),
    'resonance_score':   _s('L6_resonance_field'),
    'downstream_score':  round(
        (_s('L3_atmosphere') + _s('L5_global_electric_circuit') + _s('L6_resonance_field')) / 3, 4
    ),
    'cavity_gate_score': round(
        (_s('L4_ionosphere') + _s('L6_resonance_field')) / 2, 4
    ),
    'external_score':    _s('L0_external_drivers'),
}

print('META-SCORES')
print('=' * 65)
for name, val in meta_scores.items():
    bar = '▰' * int(val * 10) + '▱' * (10 - int(val * 10))
    print(f'  {name:<22} {bar}  {val:.3f}')

# ============================================================
# SYSTEM-STATE-KLASSIFIKATION (v2 - Meta-Score basiert)
# ============================================================

def classify_system_state(normalized, couplings, dominant, active, meta):
    """Klassifiziert Erdfeld-Zustand basierend auf Meta-Scores statt einzelner Layer"""

    avg_conf = np.mean([st['confidence'] for st in normalized.values() if st['available']])

    L0 = normalized.get('L0_external_drivers', {})
    L2 = normalized.get('L2_surface_zone', {})
    L4 = normalized.get('L4_ionosphere', {})
    L6 = normalized.get('L6_resonance_field', {})

    prep      = meta['preparation_score']    # L2
    activ     = meta['activation_score']     # L3
    electric  = meta['electric_score']       # L5
    resonance = meta['resonance_score']      # L6
    downstr   = meta['downstream_score']     # (L3+L5+L6)/3
    cavity    = meta['cavity_gate_score']    # (L4+L6)/2
    external  = meta['external_score']       # L0

    # 1. Niedrige Confidence
    if avg_conf < 0.5:
        return 'low_confidence_state', downstr, 0.4

    # 2. Geomagnetische Störung — höchste Priorität bei externem Sturm
    if L0.get('flags', {}).get('geomagnetic_storm') or L4.get('flags', {}).get('geomagnetic_storm'):
        return 'geomagnetic_disturbance_state', max(external, _s('L4_ionosphere')), 0.85

    # 3. Anomale Resonanz — L3 + L5 + L6 gemeinsam erhöht
    if activ > 0.4 and electric > 0.4 and resonance > 0.4:
        if L6.get('flags', {}).get('non_geometric_dominant') or \
           L6.get('flags', {}).get('frequency_anomaly') or \
           L6.get('flags', {}).get('amplitude_elevated'):
            return 'anomalous_resonance_state', downstr, 0.80
        return 'atmospheric_driven_resonance_state', downstr, 0.78

    # 4. Cavity Condition Shift — L4/L6 verschoben, downstream noch unbestätigt
    if cavity > 0.4 and downstr < 0.45:
        return 'cavity_condition_shift_state', cavity, 0.65

    # 5. Space Weather getrieben
    if external > 0.5 and _s('L4_ionosphere') > 0.4:
        return 'space_weather_driven_ionospheric_state', (external + _s('L4_ionosphere')) / 2, 0.80

    # 6. Gemischt — viele aktive Layer mit starken Kopplungen
    if len(active) >= 3:
        strong_couplings = [c for c in couplings if c['strength'] > 0.5]
        if len(strong_couplings) >= 2:
            return 'mixed_coupled_state', downstr, 0.70

    # 7. Seasonal Transition — L2 hoch, aber downstream schwach
    if prep > 0.45 and downstr < 0.35:
        return 'seasonal_transition_state', prep, 0.70

    # 8. ENSO-Flags als Backup für Saisonalität
    if L2.get('flags', {}).get('el_nino_developing') or L2.get('flags', {}).get('la_nina_active'):
        if downstr < 0.4:
            return 'seasonal_transition_state', prep, 0.65

    # 9. Default
    return 'normal_background_state', downstr, 0.85

system_state, state_score, state_confidence = classify_system_state(
    normalized, couplings, dominant_layer, active_layers, meta_scores
)

print(f'\nSystem-State:      {system_state}')
print(f'State-Score:       {state_score:.3f}')

# FIX: state_score_method ehrlich machen — classify_system_state liefert je
# Zustand einen anderen Basis-Score (prep / downstream / external / cavity),
# das Label darf nicht pauschal 'downstream' behaupten.
STATE_SCORE_BASIS = {
    'seasonal_transition_state':              'preparation_score',
    'geomagnetic_disturbance_state':          'max(external, L4)',
    'space_weather_driven_ionospheric_state': 'mean(external, L4)',
    'cavity_condition_shift_state':           'cavity_gate_score',
    'anomalous_resonance_state':              'downstream_score',
    'atmospheric_driven_resonance_state':     'downstream_score',
    'mixed_coupled_state':                    'downstream_score',
    'low_confidence_state':                   'downstream_score',
    'normal_background_state':                'downstream_score',
}
state_score_method = STATE_SCORE_BASIS.get(system_state, 'downstream_score')
print(f'State-Score basis: {state_score_method}')
print(f'State-Confidence:  {state_confidence:.0%}')

In [ ]:
# ============================================================
# FIELD OPERATORS — Wirkkräfte hinter den Layern
# Beschreiben WAS das System organisiert (nicht WO es ist)
# ============================================================

def _score(name):
    s = normalized.get(name, {}).get('score')
    return float(s) if s is not None else 0.0

def _metric(name, key):
    v = normalized.get(name, {}).get('key_metrics', {}).get(key)
    try: return float(v) if v is not None else None
    except: return None

def _has(*names):
    return all(normalized.get(n, {}).get('available') for n in names)

def _level(score, scheme='standard'):
    """Klassifiziert Operator-Score in qualitative Stufe"""
    if score is None: return 'unbekannt'
    if scheme == 'standard':
        if score < 0.30: return 'ruhig'
        if score < 0.60: return 'moderat'
        if score < 0.80: return 'aktiv'
        return 'stark'
    if scheme == 'transition':
        if score < 0.15: return 'keine_spannung'
        if score < 0.35: return 'schwache_übergangsspannung'
        if score < 0.55: return 'moderate_übergangsspannung'
        return 'starke_übergangsspannung'

def _placeholder(reason='keine_daten'):
    """Platzhalter-Operator wenn Daten fehlen"""
    return {
        'score':          None,
        'level':          'unbekannt',
        'interpretation': 'keine Daten verfügbar',
        'inputs':         {},
        'evidence':       [],
        'confidence':     0.0,
        'status':         reason,
    }

field_operators = {}

# ── 1. Thermal Operator (nur L2 — Vorbereitung) ──────────────
# Misst NUR thermische/feuchte Vorbereitung der Oberfläche
if _has('L2_surface_zone'):
    s2 = _score('L2_surface_zone')
    sst = abs(_metric('L2_surface_zone', 'SST_anomaly') or 0) / 2
    conv = _metric('L2_surface_zone', 'conv_pot') or 0
    score = round(min(1.0, s2*0.5 + min(1.0,sst)*0.25 + min(1.0,conv)*0.25), 4)

    evidence = []
    if s2 > 0.5: evidence.append('L2 Score erhöht')
    if sst > 0.4: evidence.append('SST-Anomalie deutlich')
    if conv > 0.5: evidence.append('Konvektionspotential hoch')

    if score > 0.6:
        interp = 'starke thermisch-feuchte Vorbereitung der Oberfläche'
    elif score > 0.3:
        interp = 'moderate thermische Vorbereitung vorhanden'
    else:
        interp = 'thermische Vorbereitung schwach'

    field_operators['thermal_operator'] = {
        'score':          score,
        'level':          _level(score),
        'interpretation': interp,
        'inputs': {
            'L2_score':        round(s2, 3),
            'SST_anomaly_norm': round(min(1.0, sst), 3),
            'conv_potential':   round(min(1.0, conv), 3),
        },
        'evidence':   evidence or ['keine auffälligen Signale'],
        'primary_driver': 'L2_surface_zone',
        'confidence': round(normalized['L2_surface_zone'].get('confidence', 0), 2),
        'status':     'modell_basiert',
    }
else:
    field_operators['thermal_operator'] = _placeholder()

# ── 2. Electric Operator (L3 → L5) ───────────────────────────
if _has('L3_atmosphere', 'L5_global_electric_circuit'):
    s3 = _score('L3_atmosphere')
    s5 = _score('L5_global_electric_circuit')
    dv = abs(_metric('L5_global_electric_circuit', 'delta_V_pct') or 0) / 30
    score = round(min(1.0, s3*0.40 + s5*0.40 + min(1.0,dv)*0.20), 4)

    evidence = []
    if s3 > 0.5: evidence.append('Gewitteraktivität erhöht')
    if s5 > 0.5: evidence.append('GEC erhöht')
    if dv > 0.3: evidence.append('Ionosphärenspannung abweichend')

    if s5 > 0.5 and s3 > 0.5:
        interp = 'starke elektrische Kopplung: Gewitter treiben GEC'
        driver = 'L3_atmosphere'
    elif s5 < 0.35:
        interp = 'GEC nahe Referenz, keine elektrische Aktivierung'
        driver = 'L5_global_electric_circuit'
    else:
        interp = 'moderate elektrische Kopplung'
        driver = 'L3_atmosphere' if s3 > s5 else 'L5_global_electric_circuit'

    field_operators['electric_operator'] = {
        'score':          score,
        'level':          _level(score),
        'interpretation': interp,
        'inputs': {
            'L3_score':       round(s3, 3),
            'L5_score':       round(s5, 3),
            'delta_V_norm':   round(min(1.0, dv), 3),
        },
        'evidence':       evidence or ['kein elektrisches Signal'],
        'primary_driver': driver,
        'confidence': round(min(
            normalized['L3_atmosphere'].get('confidence', 0),
            normalized['L5_global_electric_circuit'].get('confidence', 0)
        ), 2),
        'status':     'modell_basiert',
    }
else:
    field_operators['electric_operator'] = _placeholder()

# ── 3. Ionization Operator (L0 → L4) ─────────────────────────
if _has('L0_external_drivers', 'L4_ionosphere'):
    s0 = _score('L0_external_drivers')
    s4 = _score('L4_ionosphere')
    f107 = _metric('L0_external_drivers', 'F10.7') or 0
    f107_score = min(1.0, max(0, (f107 - 70) / 130))
    score = round(min(1.0, s0*0.35 + s4*0.45 + f107_score*0.20), 4)

    evidence = []
    L4f = normalized['L4_ionosphere'].get('flags', {})
    if L4f.get('ionospheric_disturbed'): evidence.append('Ionosphäre gestört')
    if L4f.get('radio_blackout'): evidence.append('Radio-Blackout')
    if f107_score > 0.6: evidence.append('F10.7 erhöht')
    if s0 > 0.5: evidence.append('L0 aktiv')

    if L4f.get('ionospheric_disturbed') or L4f.get('radio_blackout'):
        interp = 'Ionosphärenstörung: starker Strahlungseffekt auf Ausbreitung'
    elif f107_score > 0.6:
        interp = 'hoher Solarfluss: erhöhte D-Schicht-Ionisation'
    elif s4 < 0.3:
        interp = 'ruhige Ionosphäre, minimaler Strahlungseffekt'
    else:
        interp = 'moderate ionosphärische Modulation'

    field_operators['ionization_operator'] = {
        'score':          score,
        'level':          _level(score),
        'interpretation': interp,
        'inputs': {
            'L0_score':   round(s0, 3),
            'L4_score':   round(s4, 3),
            'F107_norm':  round(f107_score, 3),
        },
        'evidence':       evidence or ['keine Strahlungssignale'],
        'primary_driver': 'L0_external_drivers' if s0 > s4 else 'L4_ionosphere',
        'confidence': round(min(
            normalized['L0_external_drivers'].get('confidence', 0),
            normalized['L4_ionosphere'].get('confidence', 0)
        ), 2),
        'status':     'modell_basiert',
    }
else:
    field_operators['ionization_operator'] = _placeholder()

# ── 4. Geomagnetic Operator (L0 → L4/L5) ─────────────────────
if _has('L0_external_drivers'):
    s0 = _score('L0_external_drivers')
    s4 = _score('L4_ionosphere')
    s5 = _score('L5_global_electric_circuit')
    kp = _metric('L0_external_drivers', 'Kp') or 0
    bz = _metric('L0_external_drivers', 'IMF_Bz') or 0
    kp_score = min(1.0, kp / 9)
    bz_score = min(1.0, abs(bz) / 20)
    score = round(min(1.0, kp_score*0.4 + bz_score*0.2 + s0*0.2 + max(s4,s5)*0.2), 4)

    evidence = []
    L0f = normalized['L0_external_drivers'].get('flags', {})
    if L0f.get('geomagnetic_storm'): evidence.append('geomagnetischer Sturm')
    if kp_score > 0.5: evidence.append(f'Kp = {kp:.1f}')
    if bz_score > 0.5: evidence.append(f'IMF Bz = {bz:.1f} nT')

    if L0f.get('geomagnetic_storm'):
        interp = 'geomagnetischer Sturm aktiv: starke Space-Weather-Kopplung'
        driver = 'Kp_index'
    elif kp_score > 0.5:
        interp = 'erhöhte geomagnetische Aktivität'
        driver = 'Kp_index'
    elif bz_score > 0.5:
        interp = 'südwärts gerichtetes IMF Bz: offene Magnetosphäre, Energieeinschuss'
        driver = 'IMF_Bz'
    else:
        interp = 'ruhige geomagnetische Bedingungen'
        driver = 'background'

    field_operators['geomagnetic_operator'] = {
        'score':          score,
        'level':          _level(score),
        'interpretation': interp,
        'inputs': {
            'Kp_norm':    round(kp_score, 3),
            'Bz_norm':    round(bz_score, 3),
            'L0_score':   round(s0, 3),
        },
        'evidence':       evidence or ['keine geomagnetische Anregung'],
        'primary_driver': driver,
        'confidence': round(normalized['L0_external_drivers'].get('confidence', 0), 2),
        'status':     'modell_basiert',
    }
else:
    field_operators['geomagnetic_operator'] = _placeholder()

# ── 5. Resonance Model Operator (L4 + L5 + L6) ───────────────
# WICHTIG: Modell-basiert, nicht gemessen!
if _has('L6_resonance_field'):
    s4 = _score('L4_ionosphere')
    s5 = _score('L5_global_electric_circuit')
    s6 = _score('L6_resonance_field')
    ng = _metric('L6_resonance_field', 'non_geom_ratio') or 0
    score = round(min(1.0, s6*0.5 + s4*0.15 + s5*0.15 + min(1.0,ng)*0.2), 4)

    evidence = []
    L6f = normalized['L6_resonance_field'].get('flags', {})
    if L6f.get('non_geometric_dominant'): evidence.append('nicht-geometrischer Anteil dominant (Modell)')
    if L6f.get('frequency_anomaly'): evidence.append('Frequenzanomalie (Modell)')
    if L6f.get('amplitude_elevated'): evidence.append('Amplitude erhöht (Modell)')

    if L6f.get('non_geometric_dominant'):
        interp = 'modellierter nicht-geometrischer Anteil über Cavity-Geometrie hinaus'
    elif L6f.get('frequency_anomaly'):
        interp = 'modellierte Frequenzabweichung detektiert'
    elif L6f.get('amplitude_elevated'):
        interp = 'modellierte Resonanzamplitude erhöht'
    elif s6 < 0.3:
        interp = 'Resonanzfeld ruhig, schwache modellierte Modulation'
    else:
        interp = 'moderate modellierte Resonanzaktivität'

    field_operators['resonance_model_operator'] = {
        'score':          score,
        'level':          _level(score),
        'interpretation': interp,
        'inputs': {
            'L4_score':       round(s4, 3),
            'L5_score':       round(s5, 3),
            'L6_score':       round(s6, 3),
            'non_geom_ratio': round(min(1.0, ng), 3),
        },
        'evidence':       evidence or ['keine Modell-Anomalie'],
        'primary_driver': 'L6_resonance_field',
        'confidence': round(normalized['L6_resonance_field'].get('confidence', 0), 2),
        'status':     'modell_erwartet_nicht_gemessen',
    }
else:
    field_operators['resonance_model_operator'] = _placeholder()

# ── 6. Tidal/Gravity Operator (Mond-Modulation, optional) ────
field_operators['tidal_gravity_operator'] = _placeholder('keine_daten')
if _has('L1_planetary_body'):
    rv1 = (layers.get(1) or {}).get('raw_values', {})
    moon_phase = rv1.get('moon_phase')
    moon_dist  = rv1.get('moon_distance_km')
    if moon_phase is not None or moon_dist is not None:
        ts = 0.0
        inputs = {}
        evidence = []
        if moon_phase is not None:
            p = float(moon_phase) % 1
            phase_score = (1 - 4 * abs(((p + 0.25) % 0.5) - 0.25)) * 0.5
            ts += phase_score
            inputs['moon_phase'] = round(p, 3)
            if phase_score > 0.3: evidence.append('nahe Springtide (Neu/Voll)')
        if moon_dist is not None:
            d = float(moon_dist)
            dist_score = max(0, min(1.0, (406700 - d) / 50200)) * 0.5
            ts += dist_score
            inputs['moon_distance_km'] = round(d, 0)
            if dist_score > 0.3: evidence.append('Mond nahe Perigäum')
        score = round(min(1.0, max(0, ts)), 4)
        field_operators['tidal_gravity_operator'] = {
            'score':          score,
            'level':          _level(score),
            'interpretation': 'rhythmische Gezeitenmodulation durch lunare Geometrie',
            'inputs':         inputs,
            'evidence':       evidence or ['Mondkonstellation neutral'],
            'primary_driver': 'L1_planetary_body',
            'confidence':     round(normalized['L1_planetary_body'].get('confidence', 0), 2),
            'status':         'modell_basiert',
        }

# ── 7. Cross-Layer Activation Operator — DYNAMISCH ───────────
# Misst Spannung zwischen vorbereiteter und nicht-aktivierter Folgeschicht
gap_L2_L3 = max(0, _score('L2_surface_zone') - _score('L3_atmosphere'))      if _has('L2_surface_zone','L3_atmosphere') else 0
gap_L3_L5 = max(0, _score('L3_atmosphere') - _score('L5_global_electric_circuit')) if _has('L3_atmosphere','L5_global_electric_circuit') else 0
gap_L4_L6 = max(0, _score('L4_ionosphere') - _score('L6_resonance_field'))   if _has('L4_ionosphere','L6_resonance_field') else 0

gap_values = {
    'L2_to_L3': round(gap_L2_L3, 4),
    'L3_to_L5': round(gap_L3_L5, 4),
    'L4_to_L6': round(gap_L4_L6, 4),
}

# Gewichteter Score
cross_score = round(0.50*gap_L2_L3 + 0.30*gap_L3_L5 + 0.20*gap_L4_L6, 4)

# Primary Gap (welcher Übergang dominiert?)
primary_gap = max(gap_values, key=gap_values.get) if max(gap_values.values()) > 0 else 'keiner'

evidence = []
if gap_L2_L3 > 0.2: evidence.append(f'L2→L3 Gap: {gap_L2_L3:.2f} (Oberfläche vorbereitet, Atmosphäre nicht aktiviert)')
if gap_L3_L5 > 0.2: evidence.append(f'L3→L5 Gap: {gap_L3_L5:.2f} (Gewitter aktiv, GEC nicht reagiert)')
if gap_L4_L6 > 0.2: evidence.append(f'L4→L6 Gap: {gap_L4_L6:.2f} (Ionosphäre aktiv, Resonanz nicht reagiert)')

if cross_score < 0.15:
    interp = 'keine Übergangsspannung — Layer kohärent'
elif cross_score < 0.35:
    interp = f'schwache Übergangsspannung bei {primary_gap}'
elif cross_score < 0.55:
    interp = f'moderate Übergangsspannung bei {primary_gap} — Vorbereitungsphase'
else:
    interp = f'starke Übergangsspannung bei {primary_gap} — Aktivierung erwartet'

field_operators['cross_layer_activation_operator'] = {
    'score':          cross_score,
    'level':          _level(cross_score, scheme='transition'),
    'interpretation': interp,
    'inputs':         gap_values,
    'evidence':       evidence or ['keine signifikante Übergangsspannung'],
    'primary_gap':    primary_gap,
    'primary_driver': primary_gap,
    'confidence':     0.70,
    'status':         'modell_basiert',
}

# ── Operator-Vektor für Layer 8 (maschinenlesbar) ────────────
field_operator_vector = {
    name.replace('_operator', '').replace('_model', '_model'):
        (op['score'] if isinstance(op, dict) else None)
    for name, op in field_operators.items()
}

# ── Übersicht ──────────────────────────────────────────────
print('FIELD OPERATORS')
print('=' * 78)
for name, op in field_operators.items():
    if op is None or op.get('score') is None:
        status = op.get('status', 'keine_daten') if op else 'keine_daten'
        print(f'  {name:<38} —  ({status})')
    else:
        bar = '▰' * int(op['score']*10) + '▱' * (10 - int(op['score']*10))
        print(f'  {name:<38} {bar} {op["score"]:.3f}  [{op["level"]}]')
        print(f'    └─ {op["interpretation"]}')
print('=' * 78)
print('\nOperator-Vektor (für Layer 8):')
for k, v in field_operator_vector.items():
    val = f'{v:.3f}' if v is not None else 'null'
    print(f'  {k:<28} {val}')

In [ ]:
# ============================================================
# CAVITY GATE — eigener Tracking-Block
# Erkennt Cavity-Verschiebung als möglichen Vorläufer
# ============================================================

# Kopplung L4 → L6 aus couplings extrahieren
l4_l6_coupling = next(
    (c['strength'] for c in couplings
     if c['from'] == 'L4_ionosphere' and c['to'] == 'L6_resonance_field'),
    0.0
)

resonance_op_score = (field_operators.get('resonance_model_operator') or {}).get('score') or 0.0

cavity_score = meta_scores['cavity_gate_score']
cavity_active = cavity_score > 0.35

downstream_confirmation = {
    'L3': round(_s('L3_atmosphere'), 4),
    'L5': round(_s('L5_global_electric_circuit'), 4),
    'L6': round(_s('L6_resonance_field'), 4),
}
downstream_mean = round(np.mean(list(downstream_confirmation.values())), 4)

# Klassifikation des Gate-Status
if not cavity_active:
    gate_type = 'inactive'
    gate_interp = 'cavity ruhig — keine Verschiebung'
elif cavity_score > 0.5 and downstream_mean > 0.45:
    gate_type = 'confirmed_shift'
    gate_interp = 'cavity verschoben, downstream bestätigt'
elif cavity_score > 0.4 and downstream_mean < 0.30:
    gate_type = 'possible_precursor'
    gate_interp = 'cavity verschoben, downstream noch nicht aktiviert — möglicher Vorläufer'
elif cavity_score > 0.4:
    gate_interp = 'cavity verschoben, downstream-Bestätigung ausstehend'
    gate_type = 'pending_confirmation'
else:
    gate_type = 'weak'
    gate_interp = 'cavity-Verschiebung schwach'

cavity_gate = {
    'active':         cavity_active,
    'score':          round(cavity_score, 4),
    'type':           gate_type,
    'inputs': {
        'L4_score':                  round(_s('L4_ionosphere'), 4),
        'L6_score':                  round(_s('L6_resonance_field'), 4),
        'L4_to_L6_coupling':         round(l4_l6_coupling, 4),
        'resonance_model_operator':  round(resonance_op_score, 4),
    },
    'downstream_confirmation': downstream_confirmation,
    'downstream_mean':         downstream_mean,
    'interpretation':          gate_interp,
}

print('CAVITY GATE')
print('=' * 65)
print(f'  active:        {cavity_active}')
print(f'  score:         {cavity_score:.3f}')
print(f'  type:          {gate_type}')
print(f'  L4→L6 coup:    {l4_l6_coupling:.3f}')
print(f'  downstream:    L3={downstream_confirmation["L3"]:.3f}  '
      f'L5={downstream_confirmation["L5"]:.3f}  '
      f'L6={downstream_confirmation["L6"]:.3f}  (mean={downstream_mean:.3f})')
print(f'  → {gate_interp}')

---
## 6. Event-Tag-Generierung

In [ ]:

# ============================================================
# AUTOMATISCHE EVENT-TAGS — getrennt nach baseline vs signal
# ============================================================

BASELINE_TAGS = {
    'el_nino_developing', 'la_nina_active', 'sst_anomaly_high',
    'elevated_seismicity', 'non_geometric_dominance', 'high_solar_flux',
}

def gen_event_tags(layers, normalized, system_state, meta, cavity_gate):
    raw_tags = []

    L0 = layers.get(0); L1 = layers.get(1); L2 = layers.get(2)
    L3 = layers.get(3); L4 = layers.get(4); L5 = layers.get(5); L6 = layers.get(6)

    if L0 and L0.get('flags', {}).get('geomagnetic_storm'):   raw_tags.append('geomagnetic_storm')
    if L0 and L0.get('flags', {}).get('solar_flux_high'):     raw_tags.append('high_solar_flux')
    if L0 and L0.get('dominant_driver') and L0['dominant_driver'] != 'none':
        raw_tags.append('space_weather_influence')

    if L1 and L1.get('flags', {}).get('elevated_seismicity'): raw_tags.append('elevated_seismicity')
    if L1 and L1.get('flags', {}).get('strong_earthquake'):   raw_tags.append('strong_earthquake')

    if L2 and L2.get('flags', {}).get('el_nino_developing'):  raw_tags.append('el_nino_developing')
    if L2 and L2.get('flags', {}).get('la_nina_active'):      raw_tags.append('la_nina_active')
    if L2 and L2.get('flags', {}).get('thunderstorm_trigger'):raw_tags.append('surface_convection_trigger')
    if L2 and L2.get('flags', {}).get('sst_anomaly_high'):    raw_tags.append('sst_anomaly_high')

    if L3 and L3.get('flags', {}).get('active_thunderstorms'):raw_tags.append('high_thunderstorm_activity')
    if L3 and L3.get('flags', {}).get('high_cape'):           raw_tags.append('high_cape')
    if L3 and L3.get('flags', {}).get('atmospheric_instability'): raw_tags.append('atmospheric_instability')

    if L4 and L4.get('flags', {}).get('ionospheric_disturbed'):   raw_tags.append('ionospheric_disturbance')
    if L4 and L4.get('flags', {}).get('radio_blackout'):          raw_tags.append('radio_blackout')
    if L4 and L4.get('flags', {}).get('cavity_elevated'):         raw_tags.append('cavity_condition_shift')
    if L4 and L4.get('flags', {}).get('bz_southward_coupled'):    raw_tags.append('imf_bz_coupling')

    if L5 and L5.get('flags', {}).get('gec_elevated'):            raw_tags.append('gec_elevated')
    if L5 and L5.get('flags', {}).get('gec_suppressed'):          raw_tags.append('gec_suppressed')

    if L6 and L6.get('flags', {}).get('amplitude_elevated'):      raw_tags.append('elevated_resonance')
    if L6 and L6.get('flags', {}).get('frequency_anomaly'):       raw_tags.append('schumann_freq_anomaly')
    if L6 and L6.get('flags', {}).get('non_geometric_dominant'):  raw_tags.append('non_geometric_dominance')
    if L6 and L6.get('flags', {}).get('q_factor_degraded'):       raw_tags.append('q_factor_degraded')

    # Signal-Tags aus Meta-Scores
    if meta['activation_score'] > 0.4:                            raw_tags.append('l3_evening_activation')
    if meta['activation_score'] > 0.4 and meta['electric_score'] > 0.4:
        raw_tags.append('l3_l5_coupled')
    if meta['electric_score'] > 0.4 and meta['resonance_score'] > 0.4:
        raw_tags.append('l5_l6_coupled')
    if meta['downstream_score'] > 0.5:                            raw_tags.append('downstream_activated')
    if meta['preparation_score'] > 0.5 and meta['downstream_score'] < 0.3:
        raw_tags.append('preparation_without_activation')

    # Cavity-Gate-Tags
    if cavity_gate['type'] == 'possible_precursor':               raw_tags.append('cavity_shift_precursor')
    elif cavity_gate['type'] == 'confirmed_shift':                raw_tags.append('cavity_shift_confirmed')

    # Anomalous-Kandidat
    if (meta['activation_score'] > 0.35 and
        meta['electric_score']   > 0.35 and
        meta['resonance_score']  > 0.35):
        raw_tags.append('anomalous_resonance_candidate')

    # Globale Aktivität
    if meta['downstream_score'] < 0.3:  raw_tags.append('low_global_activity')
    elif meta['downstream_score'] > 0.6: raw_tags.append('high_global_activity')

    raw_tags = sorted(set(raw_tags))

    baseline_tags = sorted([t for t in raw_tags if t in BASELINE_TAGS])
    signal_tags   = sorted([t for t in raw_tags if t not in BASELINE_TAGS])
    signal_tags.append(f'state_{system_state}')

    return {
        'baseline_tags': baseline_tags,
        'signal_tags':   signal_tags,
        'all_tags':      sorted(set(baseline_tags + signal_tags)),
    }

tags_split    = gen_event_tags(layers, normalized, system_state, meta_scores, cavity_gate)
baseline_tags = tags_split['baseline_tags']
signal_tags   = tags_split['signal_tags']
event_tags    = tags_split['signal_tags']   # FIX: baseline tags raus aus event_tags

print(f'BASELINE-TAGS ({len(baseline_tags)})  — Dauerhintergrund:')
for t in baseline_tags:
    print(f'  • {t}')
print(f'\nSIGNAL-TAGS ({len(signal_tags)})  — echte Aktivierung:')
for t in signal_tags:
    print(f'  • {t}')



---
## 7. Visualisierungen

In [ ]:
# ============================================================
# LAYER-ÜBERSICHT: Score, Trend, Confidence
# ============================================================

names = list(normalized.keys())
scores = [st['score'] if st['score'] is not None else 0 for st in normalized.values()]
trend_arrows = []
for n in names:
    t = trends.get(n, {}).get('trend', 'unknown')
    trend_arrows.append({'rising':'↑','falling':'↓','stable':'→','unknown':'?'}[t])

score_colors = ['#2ecc71' if s < 0.3 else '#f39c12' if s < 0.6 else '#e74c3c' if s < 0.8 else '#c0392b'
                for s in scores]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=scores, y=names, orientation='h',
    marker_color=score_colors, opacity=0.85,
    text=[f'<b>{s:.3f}</b> {a}' for s, a in zip(scores, trend_arrows)],
    textposition='outside',
    textfont=dict(color='white', size=11)
))

fig.add_vline(x=0.3, line_dash='dot', line_color='#888780', annotation_text='ruhig')
fig.add_vline(x=0.6, line_dash='dot', line_color='#f39c12', annotation_text='moderat')
fig.add_vline(x=0.8, line_dash='dot', line_color='#e74c3c', annotation_text='aktiv')

fig.update_layout(
    title=dict(text=f'Layer-Übersicht | System-State: <b>{system_state}</b> | Conf {state_confidence:.0%}',
               font=dict(size=14)),
    xaxis=dict(title='Score [0–1]', range=[0, 1.15], gridcolor='#222244'),
    height=400, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=200, r=80, t=55, b=40), showlegend=False
)
fig.show()

In [ ]:
# ============================================================
# KOPPLUNGSMATRIX – Heatmap
# ============================================================

all_layers = list(LAYER_NAMES.values())
matrix = np.zeros((7, 7))
for c in couplings:
    i = all_layers.index(c['from'])
    j = all_layers.index(c['to'])
    matrix[i, j] = c['strength']

labels_short = [n.replace('_', '<br>', 1) for n in all_layers]

fig = go.Figure(go.Heatmap(
    z=matrix, x=labels_short, y=labels_short,
    colorscale=[[0, '#1a1a2e'], [0.3, '#534AB7'], [0.6, '#F2A623'], [1.0, '#e74c3c']],
    zmin=0, zmax=1,
    text=[[f'{v:.2f}' if v > 0 else '' for v in row] for row in matrix],
    texttemplate='%{text}',
    textfont=dict(size=10, color='white'),
    colorbar=dict(title='Stärke')
))
fig.update_layout(
    title=dict(text='Kopplungsmatrix: Wer beeinflusst wen? (Zeile → Spalte)', font=dict(size=14)),
    height=480, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=120, r=80, t=55, b=120),
    xaxis=dict(side='bottom', tickangle=30),
    yaxis=dict(autorange='reversed')
)
fig.show()

In [ ]:
# ============================================================
# LAYER-7 DASHBOARD – 4 Charts
# ============================================================

snap_current = {
    'timestamp':    RUN_TIMESTAMP,
    'system_state': system_state,
    'layers': {n: {'score': normalized[n]['score']} for n in LAYER_NAMES.values()}
}
all_snaps = history + [snap_current]
n_snaps   = len(all_snaps)

layer_list   = list(LAYER_NAMES.values())
layer_labels = [
    'L0 External Drivers',
    'L1 Planetary Body',
    'L2 Surface Zone',
    'L3 Atmosphere',
    'L4 Ionosphere',
    'L5 GEC',
    'L6 Resonance Field',
]

STATE_COLORS = {
    'normal_background_state':               '#2ecc71',
    'atmospheric_driven_resonance_state':    '#E85D24',
    'space_weather_driven_ionospheric_state':'#534AB7',
    'geomagnetic_disturbance_state':         '#e74c3c',
    'mixed_coupled_state':                  '#F2A623',
    'seasonal_transition_state':            '#639922',
    'cavity_condition_shift_state':         '#378ADD',
    'anomalous_resonance_state':            '#c0392b',
    'low_confidence_state':                 '#888780',
}
STATE_SHORT = {
    'normal_background_state':               'normal',
    'atmospheric_driven_resonance_state':    'atmospheric',
    'space_weather_driven_ionospheric_state':'space_weather',
    'geomagnetic_disturbance_state':         'geomagnetic',
    'mixed_coupled_state':                  'mixed',
    'seasonal_transition_state':            'seasonal',
    'cavity_condition_shift_state':         'cavity_shift',
    'anomalous_resonance_state':            'anomalous',
    'low_confidence_state':                 'low_conf',
}

times_str = [s['timestamp'][:16].replace('T', ' ') for s in all_snaps]

from plotly.subplots import make_subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Layer Scores Heatmap',
        'System State Timeline',
        'Layer Delta / Change  (Δ seit vorigem Snapshot)',
        'Current Dominance Ranking',
    ],
    vertical_spacing=0.20,
    horizontal_spacing=0.12,
    row_heights=[0.52, 0.48],
)

# ── Chart 1: Heatmap ─────────────────────────────────────────
snaps_heatmap = all_snaps[-10:]
times_heatmap = [s['timestamp'][:16].replace('T', ' ') for s in snaps_heatmap]

z_matrix = []
for ln in layer_list:
    row = []
    for snap in snaps_heatmap:
        v = snap.get('layers', {}).get(ln, {}).get('score')
        row.append(float(v) if v is not None else 0.0)
    z_matrix.append(row)

txt_matrix = [[f'{v:.2f}' for v in row] for row in z_matrix]

fig.add_trace(go.Heatmap(
    z=z_matrix, x=times_heatmap, y=layer_labels,
    colorscale=[[0,'#1a1a2e'],[0.25,'#378ADD'],[0.5,'#F2A623'],[0.75,'#e74c3c'],[1.0,'#c0392b']],
    zmin=0, zmax=1,
    text=txt_matrix, texttemplate='%{text}', textfont=dict(size=10, color='white'),
    showscale=True,
    colorbar=dict(len=0.42, y=0.76, x=0.46, thickness=12,
                  tickvals=[0, 0.25, 0.5, 0.75, 1.0],
                  ticktext=['0', '.25 ruhig', '.50 moderat', '.75 aktiv', '1'],
                  tickfont=dict(color='white', size=9))
), row=1, col=1)


# ── Chart 2: State Timeline ───────────────────────────────────
state_clrs  = [STATE_COLORS.get(s.get('system_state',''), '#888780') for s in all_snaps]
state_txts  = [STATE_SHORT.get(s.get('system_state',''), '?') for s in all_snaps]

fig.add_trace(go.Bar(
    x=times_str, y=[1] * n_snaps,
    marker_color=state_clrs, opacity=0.88,
    text=state_txts,
    textposition='inside',
    textfont=dict(size=10, color='white'),
    showlegend=False
), row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=2)
fig.update_xaxes(tickangle=30, row=1, col=2)

# ── Chart 3: Delta ───────────────────────────────────────────
delta_available = any(trends.get(ln, {}).get('delta_6h') is not None for ln in layer_list)

if delta_available:
    d_vals  = [trends.get(ln, {}).get('delta_6h') or 0.0 for ln in layer_list]
    d_clrs  = ['#e74c3c' if v > 0.05 else '#378ADD' if v < -0.05 else '#888780' for v in d_vals]
    d_texts = [f'{v:+.3f}' for v in d_vals]
else:
    d_vals  = [normalized[ln].get('score') or 0 for ln in layer_list]
    d_clrs  = ['#444466'] * 7
    d_texts = [f'{v:.3f} (cur)' for v in d_vals]

fig.add_trace(go.Bar(
    x=d_vals, y=layer_labels,
    orientation='h',
    marker_color=d_clrs, opacity=0.87,
    text=d_texts,
    textposition='outside', textfont=dict(color='white', size=10),
    showlegend=False
), row=2, col=1)
fig.add_vline(x=0, line_color='#555588', line_width=1, row=2, col=1)
fig.add_vline(x=0.05,  line_dash='dot', line_color='#e74c3c', line_width=1, row=2, col=1)
fig.add_vline(x=-0.05, line_dash='dot', line_color='#378ADD', line_width=1, row=2, col=1)
fig.update_xaxes(
    range=[-0.35, 0.42] if delta_available else [0, 1.1],
    title_text='Δ Score (6h)' if delta_available else 'Current Score (Δ ab 2. Snapshot)',
    gridcolor='#222244', row=2, col=1
)
if not delta_available:
    fig.add_annotation(
        x=0.55, y=6.6,
        text='⚠ Delta available after 2 snapshots',
        showarrow=False, xref='x3', yref='y3',
        font=dict(color='#F2A623', size=10),
        bgcolor='rgba(30,30,50,0.8)'
    )

# ── Chart 4: Dominance Ranking ───────────────────────────────
coup_pairs  = [f"{c['from'].split('_')[0]} → {c['to'].split('_')[0]}" for c in couplings]
coup_vals   = [c['strength'] for c in couplings]
coup_clrs   = ['#e74c3c' if v > 0.5 else '#F2A623' if v > 0.3 else '#534AB7' for v in coup_vals]
sorted_coup = sorted(zip(coup_pairs, coup_vals, coup_clrs), key=lambda x: x[1])

fig.add_trace(go.Bar(
    x=[v for _, v, _ in sorted_coup],
    y=[l for l, _, _ in sorted_coup],
    orientation='h',
    marker_color=[c for _, _, c in sorted_coup], opacity=0.88,
    text=[f'{v:.2f}' for _, v, _ in sorted_coup],
    textposition='outside', textfont=dict(color='white', size=10),
    showlegend=False
), row=2, col=2)
for t, col, label in [(0.25,'#555588','ruhig'),(0.50,'#888780','moderat'),(0.75,'#e74c3c','stark')]:
    fig.add_vline(x=t, line_dash='dot', line_color=col, line_width=1,
                  annotation_text=label,
                  annotation_font=dict(color=col, size=8),
                  annotation_position='top',
                  row=2, col=2)
fig.update_xaxes(range=[0, 1.12], title_text='Kopplungsstärke', gridcolor='#222244', row=2, col=2)

# ── Global State Header ───────────────────────────────────────
fig.add_annotation(
    x=0.5, y=1.06, xref='paper', yref='paper',
    text=(
        f'<b>Earth Field State:</b> {STATE_SHORT.get(system_state, system_state)}  '
        f'  <b>Dominant Layer:</b> {dominant_layer.split("_",1)[-1].replace("_"," ").title()}  '
        f'  <b>Score:</b> {state_score:.3f}'
        f'  <b>Confidence:</b> {state_confidence:.0%}'
        f'  <b>Snapshots:</b> {n_snaps}'
    ),
    showarrow=False, font=dict(size=12, color='white'),
    bgcolor='rgba(30,30,60,0.85)',
    bordercolor='#534AB7', borderwidth=1,
    xanchor='center'
)

# ── Layout ────────────────────────────────────────────────────
fig.update_layout(
    height=680,
    plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=150, r=80, t=100, b=70),
    showlegend=False,
    font=dict(color='white', size=11),
)
for r, c in [(1,1),(1,2),(2,1),(2,2)]:
    fig.update_yaxes(gridcolor='#222244', tickfont=dict(color='white', size=10), row=r, col=c)
    fig.update_xaxes(gridcolor='#222244', tickfont=dict(color='white', size=9), row=r, col=c)

fig.show()


In [ ]:
# ============================================================
# FIELD OPERATORS – Chart
# ============================================================

# Nur Operatoren mit gültigem Score (Platzhalter herausfiltern)
op_items = [(n, v) for n, v in field_operators.items()
            if v is not None and v.get('score') is not None]

op_names  = [n for n, _ in op_items]
op_scores = [v['score'] for _, v in op_items]
op_levels = [v.get('level', '') for _, v in op_items]

op_colors = ['#2ecc71' if s < 0.3 else '#f39c12' if s < 0.6 else '#e74c3c'
             for s in op_scores]

# Saubere Labels
op_labels = [n.replace('_operator', '').replace('_', ' ').title() for n in op_names]

fig = go.Figure(go.Bar(
    x=op_scores, y=op_labels,
    orientation='h', marker_color=op_colors, opacity=0.85,
    text=[f'{s:.3f}  [{lvl}]' for s, lvl in zip(op_scores, op_levels)],
    textposition='outside',
    textfont=dict(color='white', size=11),
    hovertemplate='<b>%{y}</b><br>Score: %{x:.3f}<extra></extra>',
))

# Schwellen-Linien
fig.add_vline(x=0.3, line_dash='dot', line_color='#888780',
              annotation_text='ruhig', annotation_font=dict(color='#888780', size=9))
fig.add_vline(x=0.6, line_dash='dot', line_color='#f39c12',
              annotation_text='moderat', annotation_font=dict(color='#f39c12', size=9))
fig.add_vline(x=0.8, line_dash='dot', line_color='#e74c3c',
              annotation_text='aktiv', annotation_font=dict(color='#e74c3c', size=9))

# Hinweis falls Operatoren fehlen
n_missing = len(field_operators) - len(op_items)
subtitle = f' ({n_missing} ohne Daten)' if n_missing > 0 else ''

fig.update_layout(
    title=dict(text=f'Field Operators — Wirkkräfte hinter den Layern{subtitle}',
               font=dict(size=14)),
    xaxis=dict(title='Operator Score [0–1]', range=[0, 1.15], gridcolor='#222244'),
    height=360, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=220, r=80, t=55, b=40), showlegend=False,
)
fig.show()

---
## 8. Export & Archiv

In [ ]:
# ============================================================
# AKTUELLER ZUSTAND – layer7_state.json
# ============================================================

# AGGREGATE_EXCLUDE stammt aus der Dominanz-Zelle: Meso ist ein Mediator-Proxy
# und bleibt aus schichtuebergreifenden Mittelwerten heraus (Begruendung dort).
_agg = {n: st for n, st in normalized.items() if n not in AGGREGATE_EXCLUDE}
avg_score      = round(float(np.mean([st['score'] for st in _agg.values() if st['score'] is not None])), 4)
avg_confidence = round(float(np.mean([st['confidence'] for st in _agg.values() if st['available']])), 4)

earth_field_state = {
    'timestamp':       RUN_TIMESTAMP,
    'engine_version':  '2.0',
    'layer':           7,
    'name':            'Earth Field State Engine',

    # Globaler Zustand
    'system_state':          system_state,
    'state_score':           round(float(state_score), 4),
    'state_score_method':    state_score_method,
    'state_confidence':      round(float(state_confidence), 2),
    'avg_score':             avg_score,
    'avg_score_method':      'mean_of_available_layers_excl_mediator_proxies',
    'aggregate_excluded':    sorted(AGGREGATE_EXCLUDE),   # Semantik im Snapshot festhalten
    'avg_confidence':        avg_confidence,

    # Meta-Scores (neu)
    'meta_scores': meta_scores,

    # ENSO Kontext
    'enso_context': enso_context,

    # State-Confidence Begründung (meta-basiert)
    'state_confidence_reason': (
        f'downstream_score={meta_scores["downstream_score"]:.3f} '
        f'(L3={meta_scores["activation_score"]:.3f}, '
        f'L5={meta_scores["electric_score"]:.3f}, '
        f'L6={meta_scores["resonance_score"]:.3f}). '
        f'preparation_score={meta_scores["preparation_score"]:.3f}. '
        + ('Downstream bestätigt.' if meta_scores['downstream_score'] > 0.4
           else 'Downstream schwach — Vorbereitung ohne Aktivierung.')
    ),

    # Dominanz-Analyse
    'dominance': {
        'dominant_layer':   dominant_layer,
        'secondary_layer':  secondary_layer,
        'active_layers':    active_layers,
        'weak_layers':      weak_layers,
    },

    # Pro Layer
    'layers': {
        name: {
            **st,
            **trends.get(name, {})
        }
        for name, st in normalized.items()
    },

    # Cavity Gate (neu)
    'cavity_gate': cavity_gate,

    # Diagnostische Features
    'diagnostic_features': (lambda: {
        'surface_atmosphere_gap':    round(
            (normalized['L2_surface_zone']['score'] or 0) -
            (normalized['L3_atmosphere']['score'] or 0), 4),
        'surface_to_resonance_gap':  round(
            (normalized['L2_surface_zone']['score'] or 0) -
            (normalized['L6_resonance_field']['score'] or 0), 4),
        'atmosphere_to_gec_gap':     round(
            (normalized['L3_atmosphere']['score'] or 0) -
            (normalized['L5_global_electric_circuit']['score'] or 0), 4),
        'external_pressure_low':     bool(
            (normalized['L0_external_drivers']['score'] or 1) < 0.3),
        'resonance_confirmed':       bool(
            meta_scores['resonance_score'] > 0.4 and
            meta_scores['activation_score'] > 0.4),
        'transition_state':          bool(
            meta_scores['preparation_score'] > 0.4 and
            meta_scores['downstream_score'] < 0.35),
        'interpretation': (
            'surface_prepared_but_atmosphere_not_activated'
            if meta_scores['preparation_score'] > 0.4 and
               meta_scores['downstream_score'] < 0.35
            else 'coupled_activation'
            if meta_scores['downstream_score'] > 0.4
            else 'background_state'
        ),
    })(),

    # Kp-Kontext
    'kp_context': {
        'L0_Kp_used':    normalized['L0_external_drivers']['key_metrics'].get('Kp'),
        'L4_Kp_current': normalized['L4_ionosphere']['key_metrics'].get('Kp'),
        'note': (
            'Different Kp windows/sources: '
            'L0 uses broader context (Layer-0-Score-Kp), '
            'L4 uses current 1-min ionospheric state.'
        ),
    },

    # Kopplungen
    'couplings': couplings,

    # Field Operators
    'field_operators':       field_operators,
    'field_operator_vector': field_operator_vector,

    # Tags — getrennt nach Typ (neu)
    'baseline_tags': baseline_tags,
    'signal_tags':   signal_tags,
    'event_tags':    event_tags,   # Rückwärtskompatibilität für Layer 8

    # Schwellenwerte
    'thresholds': {
        'level_thresholds': LEVEL_THRESHOLDS,
        'trend_threshold':  0.05,
        'active_layer':     0.5,
        'weak_layer':       0.3,
        'meta_downstream_activated': 0.5,
        'meta_preparation_threshold': 0.45,
        'cavity_gate_threshold': 0.35,
    },

    # Layer-7-Output für Layer 8
    'layer8_handoff': {
        'state_summary': (
            f'System-State: {system_state} | Score {state_score:.3f} | '
            f'Conf {state_confidence:.0%} | Dom: {dominant_layer} | '
            f'Downstream: {meta_scores["downstream_score"]:.3f} | '
            f'CavityGate: {cavity_gate["type"]} | '
            f'Signal-Tags: {len(signal_tags)} | Baseline-Tags: {len(baseline_tags)} | '
            f'History: {len(history)+1} snapshots'
        ),
        'research_questions': [
            'Welche Signal-Tag-Kombinationen treten vor anomalous_resonance_state auf?',
            'Wie entwickelt sich cavity_gate_type über Zeit?',
            'Wann folgt auf cavity_shift_precursor eine vollständige Aktivierung?',
            'Korreliert downstream_score mit ΔL3 am Abend?',
        ],
    },
}

# numpy-Bereinigung
def _to_python(obj):
    if isinstance(obj, dict):  return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [_to_python(v) for v in obj]
    if isinstance(obj, np.bool_):    return bool(obj)
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return None if np.isnan(obj) else float(obj)
    return obj
earth_field_state = _to_python(earth_field_state)
from atmosphere.meta.role_proxy_writeback import annotate_l7
earth_field_state = annotate_l7(earth_field_state)
with open(STATE_FILE, 'w', encoding='utf-8') as f:
    json.dump(earth_field_state, f, indent=2, ensure_ascii=False)
print(f'✅ {STATE_FILE} gespeichert')

# ============================================================
# HISTORIE-ARCHIV – layer7_history.jsonl
# ============================================================

with open(HISTORY_FILE, 'a', encoding='utf-8') as f:
    f.write(json.dumps(earth_field_state, ensure_ascii=False) + '\n')
print(f'✅ Snapshot an {HISTORY_FILE} angehängt (Archiv für Layer 8)')
print(f'   Archivgröße: {len(history)+1} Snapshots')

# Kompakte Zusammenfassung
print('\n' + '=' * 78)
print('EARTH FIELD STATE – ZUSAMMENFASSUNG')
print('=' * 78)
print(f'  Timestamp:          {RUN_TIMESTAMP}')
print(f'  System-State:       {system_state}')
print(f'  State-Score:        {state_score:.3f}  ({avg_confidence:.0%} confidence)')
print(f'  Dominant Layer:     {dominant_layer}')
print(f'  Active Layers:      {active_layers}')
print(f'  Downstream-Score:   {meta_scores["downstream_score"]:.3f}  '
      f'(L3={meta_scores["activation_score"]:.3f} '
      f'L5={meta_scores["electric_score"]:.3f} '
      f'L6={meta_scores["resonance_score"]:.3f})')
print(f'  Preparation-Score:  {meta_scores["preparation_score"]:.3f}  (L2)')
print(f'  ENSO:               {enso_context["phase_class"]}  '
      f'({enso_context["event_risk"]})  '
      f'Niño3.4={enso_context["nino34_anomaly_degC"]}°C')
print(f'  Cavity Gate:        {cavity_gate["type"]}  (score={cavity_gate["score"]:.3f})')
print(f'  Baseline-Tags:      {baseline_tags}')
print(f'  Signal-Tags:        {signal_tags}')
print(f'  Strongest Coupling: {max(couplings, key=lambda c: c["strength"])["from"]} → '
      f'{max(couplings, key=lambda c: c["strength"])["to"]} '
      f'({max(c["strength"] for c in couplings):.2f})')
print(f'  History Archive:    {len(history)+1} snapshots in {HISTORY_FILE}')
print('=' * 78)

---
## Zusammenfassung Layer 7

| Aspekt | Inhalt |
|--------|--------|
| **Rolle** | Earth Field State Engine – Zustandsmaschine + Kopplungsmodell + Datenarchiv |
| **Input** | `layer{0..6}_state.json` |
| **Output (aktuell)** | `layer7_state.json` – kompletter Systemzustand |
| **Output (Archiv)** | `layer7_history.jsonl` – ein Snapshot pro Engine-Lauf |
| **Systemzustände** | 9 Klassen: normaler_Hintergrund, atmosphärisch_getriebene_Resonanz, weltraumwetterbedingte_Ionenosphäre, geomagnetische_Störung, gemischte_Kopplung, Verschiebung_der_Hohlraum-Bedingung, saisonaler_Übergang, anomale_Resonanz, geringe_Zuverlässigkeit |
| **Meta-Scores** | 6 aggregierte Indikatoren: Downstream, Vorbereitung, Aktivierung, Elektrik, Resonanz, Cavity_Gate (jeweils 0–1) |
| **Kopplungen** | 7 definierte Pfade (L0→L4, L3→L5, L3→L6, L4→L6, L5→L6, L0→L5, L2→L3) |
| **Trends** | Δ1h, Δ6h, Δ24h, Volatilität (basierend auf Historie) |
| **Cavity Gate** | Dedizierter Überwachungsblock — 5 Typen: inaktiv, schwach, möglicher_Vorläufer, Bestätigung_ausstehend, Verschiebung_bestätigt |
| **Feldoperatoren** | 7 Operatoren: thermisch, elektrisch, Ionisierung, geomagnetisch, Resonanzmodell, Gezeiten-Gravitation, schichtenübergreifende_Aktivierung |
| **Ereignis-Tags** | 3-fache Aufteilung: Baseline-Tags (persistenter Hintergrund), Signal-Tags (tatsächliche Aktivierung), Ereignis-Tags (kombiniert, für Kompatibilität mit Schicht 8) |
| **Übergabe an Schicht 8** | Strukturierter String mit Zustandszusammenfassung + 4 vorformulierte Forschungsfragen |
| **→ Schicht 8** | Forschung zu Historie, Tag-Mustern, Kopplungskorrelationen, Entwicklung des Cavity Gate |

> **Wichtig:** Bei jedem Engine-Lauf wird ein neuer Snapshot an `layer7_history.jsonl` angehängt. Layer 8 wird auf diesem wachsenden Archiv arbeiten.

> **Nächster Schritt:** `atmosphere_analysis_layer8.ipynb` – Research / Hypothesen / Systemfragen